In [1]:
# ─── 1) Imports & Setup ──────────────────────────────────────────
import os
import json
import glob
from tqdm import tqdm
import numpy as np
from skimage import io, img_as_ubyte
from skimage.measure import regionprops
from pycocotools import mask as mask_utils
import collections

# ─── 2) Paths & parameters ─────────────────────────────────────────────
BASE_DIR          = r'C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\image and seg data'
OUTPUT_PATCH_ROOT = r'C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\processed data\patches_2508'
COCO_JSON_PATH    = r'C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\processed data\json_2508'

# Ensure output directories exist
os.makedirs(OUTPUT_PATCH_ROOT, exist_ok=True)
os.makedirs(COCO_JSON_PATH, exist_ok=True)

PATCH_SIZE    = (256, 256)
STRIDE        = 128
MIN_MASK_AREA = 50
IOU_THRESHOLD = 0.3  # Adjusted IOU threshold for rbc/cbc labelling
TARGET_SHAPE  = (1024, 1024)

# ─── 3) Class mapping ──────────────────────────────────────────────────
CLASS_MAP = {'other': 0, 'rbc': 1, 'cbc': 2}
categories = [{'id': v, 'name': k} for k, v in CLASS_MAP.items()]

# ─── 4) File discovery and grouping ────────────────────────────────────
# Code for file discovery remains the same...
img_files = glob.glob(os.path.join(BASE_DIR, '*.png')) + glob.glob(os.path.join(BASE_DIR, '*.jpg'))
seg_files = glob.glob(os.path.join(BASE_DIR, '*_seg.npy'))
file_groups = collections.defaultdict(dict)
for f in img_files:
    basename = os.path.basename(f)
    parts = basename.split('-')
    channel, img_id = parts[0], parts[1].split('.')[0]
    file_groups[img_id][channel] = f
for f in seg_files:
    basename = os.path.basename(f)
    parts = basename.split('-')
    channel, img_id = parts[0], parts[1].split('_seg.')[0]
    file_groups[img_id][f'{channel}_seg'] = f
print(f"Found {len(file_groups)} image groups to process.")


# ─── 5) Helper Functions ───────────────────────────────────────────────
def load_segmentation_mask(filepath):
    try:
        data = np.load(filepath, allow_pickle=True)
        if data.ndim == 0 and isinstance(data.item(), dict) and 'masks' in data.item():
            return data.item()['masks']
        elif isinstance(data, np.ndarray): return data
        else: return None
    except Exception: return None

def pad_image_to_target(image, target_shape):
    th, tw = target_shape
    if image.ndim == 3: h, w, c = image.shape
    elif image.ndim == 2: h, w = image.shape
    else: return image
    pad_bottom, pad_right = max(0, th - h), max(0, tw - w)
    if image.ndim == 3:
        padded_image = np.pad(image, ((0, pad_bottom), (0, pad_right), (0, 0)), mode='constant')
    else:
        padded_image = np.pad(image, ((0, pad_bottom), (0, pad_right)), mode='constant')
    return padded_image

def calculate_iou(mask1, mask2):
    intersection = np.logical_and(mask1, mask2)
    union = np.logical_or(mask1, mask2)
    return np.sum(intersection) / np.sum(union) if np.sum(union) > 0 else 0


# ─── 6) Main processing loop with FEATURE ENGINEERING ────────────────
patch_coco_output = {'images': [], 'annotations': [], 'categories': categories}
patch_img_id_counter = 0
patch_ann_id_counter = 0

for img_id, paths in tqdm(file_groups.items()):
    if not all(k in paths for k in ['C1', 'C1_seg', 'C2_seg', 'C3_seg']): continue

    img_c1 = io.imread(paths['C1'])
    seg_c1 = load_segmentation_mask(paths['C1_seg'])
    seg_c2 = load_segmentation_mask(paths['C2_seg'])
    seg_c3 = load_segmentation_mask(paths['C3_seg'])
    
    if any(x is None for x in [img_c1, seg_c1, seg_c2, seg_c3]): continue

    img_c1 = pad_image_to_target(img_c1, TARGET_SHAPE)
    seg_c1 = pad_image_to_target(seg_c1, TARGET_SHAPE)
    seg_c2 = pad_image_to_target(seg_c2, TARGET_SHAPE)
    seg_c3 = pad_image_to_target(seg_c3, TARGET_SHAPE)
    
    # Make sure img_c1 is grayscale for intensity calculations
    if img_c1.ndim == 3:
        img_c1 = img_c1.mean(axis=2).astype(np.uint8)

    mask_props = []
    # ★★★★★★★★★★★★★★★★★★★★★★★★★ FEATURE CALCULATION STARTS HERE ★★★★★★★★★★★★★★★★★★★★★★★★★
    # Pass the intensity image (img_c1) to regionprops to calculate pixel-based features
    props_c1 = regionprops(seg_c1, intensity_image=img_c1)
    
    masks_c2 = [(seg_c2 == i) for i in np.unique(seg_c2) if i > 0]
    masks_c3 = [(seg_c3 == i) for i in np.unique(seg_c3) if i > 0]
    
    for prop in props_c1:
        mask_c1 = (seg_c1 == prop.label)
        
        # --- Classification logic (as before) ---
        max_iou_c2 = max([calculate_iou(mask_c1, m) for m in masks_c2], default=0)
        max_iou_c3 = max([calculate_iou(mask_c1, m) for m in masks_c3], default=0)
        
        class_id = CLASS_MAP['other']
        if max_iou_c2 > IOU_THRESHOLD and max_iou_c2 >= max_iou_c3: class_id = CLASS_MAP['rbc']
        elif max_iou_c3 > IOU_THRESHOLD and max_iou_c3 > max_iou_c2: class_id = CLASS_MAP['cbc']
        
        # --- Engineer and store new features ---
        min_y, min_x, max_y, max_x = prop.bbox
        bbox_height = max_y - min_y
        bbox_width = max_x - min_x
        
        aspect_ratio = bbox_width / bbox_height if bbox_height > 0 else 0
        circularity = (4 * np.pi * prop.area) / (prop.perimeter ** 2) if prop.perimeter > 0 else 0
        
        # Get pixel values only within the mask and calculate standard deviation
        pixels_in_mask = prop.intensity_image[prop.image]
        std_intensity = np.std(pixels_in_mask) if pixels_in_mask.size > 0 else 0
        
        mask_props.append({
            'mask': mask_c1,
            'class_id': class_id,
            'bbox': prop.bbox,
            # Add all new features to be passed to the final annotations
            'features': {
                'original_area': prop.area,
                'aspect_ratio': aspect_ratio,
                'circularity': circularity,
                'solidity': prop.solidity,
                'mean_intensity': prop.mean_intensity,
                'std_intensity': std_intensity
            }
        })
    # ★★★★★★★★★★★★★★★★★★★★★★★★★★★ FEATURE CALCULATION ENDS HERE ★★★★★★★★★★★★★★★★★★★★★★★★★★

    # --- Patching loop (as before) ---
    img_h, img_w = img_c1.shape[:2]
    patch_h, patch_w = PATCH_SIZE
    
    for patch_y in range(0, img_h, STRIDE):
        for patch_x in range(0, img_w, STRIDE):
            y_start, y_end = patch_y, min(patch_y + patch_h, img_h)
            x_start, x_end = patch_x, min(patch_x + patch_w, img_w)
            
            if (y_end - y_start) != patch_h or (x_end - x_start) != patch_w: continue
            
            img_patch = img_c1[y_start:y_end, x_start:x_end]
            patch_has_annotations = False
            current_patch_annotations = []
            
            for mask_prop in mask_props:
                mask_in_patch = mask_prop['mask'][y_start:y_end, x_start:x_end]
                if np.sum(mask_in_patch) > MIN_MASK_AREA:
                    patch_has_annotations = True
                    fortran_mask = np.asfortranarray(mask_in_patch)
                    rle = mask_utils.encode(fortran_mask)
                    rle['counts'] = rle['counts'].decode('utf-8')
                    
                    ann = {
                        'id': patch_ann_id_counter, 'image_id': patch_img_id_counter,
                        'category_id': mask_prop['class_id'], 
                        'bbox': mask_utils.toBbox(rle).tolist(),
                        'original_bbox': list(mask_prop['bbox']),
                        'area': float(mask_utils.area(rle)),
                        'segmentation': rle, 'iscrowd': 0,
                    }
                    # Embed the pre-calculated features for the original mask
                    ann.update(mask_prop['features'])
                    
                    current_patch_annotations.append(ann)
                    patch_ann_id_counter += 1

            if patch_has_annotations:
                patch_filename = f'patch_{patch_img_id_counter:06d}.png'
                io.imsave(os.path.join(OUTPUT_PATCH_ROOT, patch_filename), img_as_ubyte(img_patch))
                
                patch_coco_output['images'].append({
                    'id': patch_img_id_counter, 'file_name': patch_filename,
                    'height': patch_h, 'width': patch_w
                })
                
                for ann in current_patch_annotations:
                    ann['image_id'] = patch_img_id_counter
                patch_coco_output['annotations'].extend(current_patch_annotations)
                patch_img_id_counter += 1

# ─── 7) Save the final COCO JSON file ──────────────────────────────────
FINAL_JSON_PATH = os.path.join(COCO_JSON_PATH, 'patches_coco_with_features.json')
with open(FINAL_JSON_PATH, 'w') as f:
    json.dump(patch_coco_output, f, indent=4)

print(f"\nProcessing complete!")
print(f"Feature-rich JSON file saved to: {FINAL_JSON_PATH}")

Found 18 image groups to process.


  0%|          | 0/18 [00:00<?, ?it/s]c:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\.venv\lib\site-packages\skimage\_shared\utils.py:328: UserWarning: C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\processed data\patches_2508\patch_000022.png is a low contrast image
  return func(*args, **kwargs)
c:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\.venv\lib\site-packages\skimage\_shared\utils.py:328: UserWarning: C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\processed data\patches_2508\patch_000023.png is a low contrast image
  return func(*args, **kwargs)
c:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\.venv\lib\site-packages\skimage\_shared\utils.py:328: UserWarning: C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\processed data\patches_2508\patch_000028.png is a low contrast image
  return func(*args, **kwargs)
c:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\.venv\lib\site-packages\ski


Processing complete!
Feature-rich JSON file saved to: C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\processed data\json_2508\patches_coco_with_features.json
